# MDB Vegetation Vulnerability

Modified from the MDBA BWS Vulnerabilities Project

The BWS Priorities Project aimed to spatially and temporally summarise metrics of vulnerability (combining condition and stress) for vegetation and waterbirds in the Murray-Darling Basin with the aim of informing the setting of annual watering priorities for these target groups.

**Project report**: Hale, J., Brooks, S., Campbell, C. and McGinness, H. (2023) Assessing Vulnerability for use in Determining Basin-scale Environmental Watering Priorities. A Report to the Commonwealth Environmental Water Office, Canberra.

* [LInk to report from the DCCEEW website](https://www.dcceew.gov.au/sites/default/files/documents/assessing-vulnerability-use-determining-basin-scale-environmental-watering-priorities.pdf)

The Jupyter notebook is the final stage of data processing that pulls together multiple data sets to summarise and score the condition metrics, stress metrics and then add the scores to the final vulnerability metric.  Multiple input data files are read in, pivoted to tabular format with years as columns.  The measurement of vulnerability relies on first calculating the long-term baseline (mean of all years excluding the millennium drought) then scoring the deviation from the baseline.   Metrics calculated for ANAE ecosystem polygons are aggregated together as an area weighted average for larger spatial units (e.g. Ramsar sites, valleys).

## Data Inputs

 1. Australian National Aquatic Ecosystem (ANAE) mapping  v3 - The ANAE identifies different vegetation types and provides the spatial units used to summarise other data. Polygons < 1 Ha are removed as they are too small to meet the reliability requirements of the WIT tool and MODIS derived NDVI.  
 1. Geoscience Australia Wetland Insights Tool (WIT) - WIT data observations for all ANAE polygons > 1 Ha in the MDB 1986-present.  Raw data supplied by Geoscience Australia for individual observation dates through the Landsat Record summarised into daily, yearly, all-time and inundation event statistics (a separate jupyter notebook)
 1. Normalized Difference Vegetation Index (NDVI) - Average NDVI per ANAE polygon per year 1986-present calculated using google earth engine reducer: shared code: [(Flow-MER GoogleEarthEngine_scripts)](https://github.com/Flow-MER/GoogleEarthEngine_scripts)
 1. [Root Zone Soil Moisture (Australian Water Outlook)](https://awo.bom.gov.au/products/historical/soilMoisture-rootZone) - Mean root zone soil moisture per ANAE polygon per year was generated using ArcGIS but there are many ways to calculate the annual average per polygon from the AWO netcdf   
 1. Stress thresholds for vegetation based on durations since last inundation for different functional groups that were identified by experts are coded directly into this Jupyter Notebook

## Data Outputs

This notebook writes the various metric to the working directory in tabular format csv files (spatial units in rows, years in columns) that can be read by Microsoft Excel.  Baseline values and scores are added to the tables as additional columns. The output includes spatial scales that were not included in the BWS Vulnerabilities project report but may be useful for other investigations or to inform water planning at those locations (e.g. DIWA and Ramsar sites)

Output files for habitat metrics follow the naming convention: {metric}_{aggregator}_{year_window_width}yr_condition.csv
e.g.  pv_median_DIWA_5yr_condition.csv  is the median "pv" (green fractional cover) with ANAE polygons aggregated to larger DIWA wetland scales using a 5-year moving window in which to calculate rates of change.  

*NOTE:  The outputs generated from this notebook will vary from the report because this code has removed the MDBA Stand Condition tool inputs and made improvements to the NDVI inputs

### Mapping the outputs

* Patterns can be visualised in GIS by joining the output files to the relevant spatial layers.  Many of the vegetation maps in the report used the ANAE polygons scale to visualise the patterns - this was done by joining **FINAL_BWSVulnerability_vegetation_ANAE.csv** to the **ANAEv3** using the **UID** polygon identifier.  Mapping whole Valley aggregated scores would be done by joining **FINAL_BWSVulnerability_vegetation_Valley.csv** to **BWSRegions.shp** using the **BWS_Region**.

## Processing Environment

Python 3.11.11

install requirements

```pip install -r requirements.txt```

## Repeating or extending the analysis to additional years of data

Extending the analysis requires:

1. collating new input data and appending to the current 1986-2024 source files
1. edit the definition of the **alltime** variable to extend past 2024.
1. re-run the notebook

Source data comes from a variety of places and requires a different technologies to assemble as outlined above.  The current source files should be used as the template to append to,  which should ensure the updated files will run with this workbook.  There is some additional code built into the workbook to re-build spatial relationships among data

The code was built to test the method within the confines of a project so it isn't always pretty.    If the logic is not clear please refer to the report and reach out to the report authors with questions.

***

## Contact

Dr Shane Brooks
<https://brooks.eco>

![Brooks.eco logo](brooks-logo.png "Brooks Ecology & Technology")


# Load packages
Import Python packages that are used for the analysis.

Use standard import commands; some are shown below. 


In [1]:
import os
import sys

import pandas as pd
import numpy as np
import geopandas as gpd
from tqdm.notebook import tqdm


# User Defined Parameters

In [2]:
# set the path to the spatial data (shape files)
spatial_path = "./input/spatial/"


# set the path to the data input files
data_path = "./input/csv"

#  set the working directory
out_path = "./output/"

In [3]:
# define year ranges used by the method

alltime = list(
    range(1987, 2025)
)  # 1986 is excluded as it has incomplete data.  Last data year is 2024
millennium_drought = list(
    range(2001, 2010)
)  # excludes 2010 because last year in a range is not included
# yearcols is a dictionary to translate numberic year column totals of pivot tables into strings as required for shapefiles and csv headers
yearcols = {y: "y" + str(y) for y in alltime}

# VEGETATION
veg_window_width = (
    5  # veg condition/stress is averaged over a moving 5 year window up to a given year
)
veg_trend_width = 2  # veg metric trends are measured in the most recent 2 years leading up to a given year


# Vegetation thresholds noting that some ranges are imprecise and have gaps.
# modified from BWS project as follows:
#       # added chenopod shrubland  with threshold 0-4-10
#       # added cooba with threshold 0-4-7.


# the three bins are defined in python using three numbers [0. threshold#1, threshold2]

# and scored as:  0 > LOW > threshold#1 > MEDIUM > threshold#2 > HIGH


# Vegetation stress thresholds based on the time-since-last-inundation (tsli)
# Threshold scores are set for three bins (LOW, MEDIUM and HIGH stress)
vegetation_tsli_stress_thresholds = {
    # RRG swamps forests and woodlands 1-2 years, 3-4 years, ≥ 5 years
    "river red gum swamps and forests": [0, 730, 1825],
    "river red gum woodland": [0, 730, 1825],
    # Black box, 3 – 4 years, 5 – 6 years, ≥ 7 years
    "black box": [0, 1460, 2555],
    # Cooba  1-4 years, 5-6 years, ≥ 7 years
    "cooba": [0, 1460, 2555],
    # Coolibah, 10 years, 20 years, > 20 years
    "coolibah": [0, 3650, 7300],
    # Lignum, 3 years, 4 years, ≥ 7 years
    "lignum": [0, 1095, 2555],
    # chenopods/shrubland, 1-3 years, 4-10 years, ≥ 10 years
    "shrubland": [0, 1095, 3650],
    # Submerged vegetation, < 3 months, 3 – 4 months, > 4 months
    "submerged lake": [0, 90, 120],
    # Tall reeds, < 1 year, 1 – 2 years, > 2 years
    "tall reed beds": [0, 365, 730],
    # Grassy meadows, < 8 months, 8 – 10 months, > 10 months
    "grassy meadows": [0, 240, 300],
    # Herb fields, 1 year, 2 – 4 years, > 4 years
    "herbfield": [0, 365, 1460],
    # not vegetated was used for waterbirds  10 years, 20 years, > 20 years
    # "clay pan": [0,3650,7300],
}

#these global dictionaries are redefined later in the code when spatial data is loaded - here for code development purposes
aggregators = {}
aggfields = {}

# Function definitons

In [4]:
def pivot_year(df, wit_metric, pkey="UID"):
    """
        Pivot the input data to a dataframe with years as column headers
        calculate the mean and stddev for the baseline years =1989-2022 excluding the millennium drought

        count =  number of years with waterbird counts
        baseline = mean of baseline years
        max = maximum value for baseline period
        median = median value for baseline period
        mad =  median absolute deviation baseline
               mad is a non-parametric standard deviation used with the
               waterbird data because there are lots of spatial units with no or few count records
    '''


    """
    allyears = df["year"].unique().tolist()
    baseline = [y for y in allyears if y not in millennium_drought]
    
    # Create pivot table and calculate statistics in one go
    df_pivot = df.pivot(index=pkey, columns="year", values=wit_metric)
    baseline_data = df_pivot[baseline]
    
    # Calculate all statistics at once
    df_count = df_pivot.count(axis=1, numeric_only=True).rename("count")
    df_baseline = baseline_data.mean(axis=1, numeric_only=True).rename("baseline")
    df_stddev = baseline_data.std(axis=1, numeric_only=True).rename("stddev")
    df_max = baseline_data.max(axis=1, numeric_only=True).rename("max")
    df_median = baseline_data.median(axis=1, numeric_only=True).rename("median")
    
    # Calculate MAD more efficiently - avoid creating multiple dataframes
    deviations = baseline_data.sub(df_median, axis=0).abs()
    df_mad = deviations.median(axis=1, numeric_only=True).rename("mad")
    
    # Combine all results
    return pd.concat([df_pivot, df_count, df_baseline, df_stddev, df_max, df_median, df_mad], axis=1)

def fn_slope(d):
    """
    calculate the rate of change as the slope through the supplied points
    input is a series of values
    output is the rate of change
    """
    yvalues = d.values
    if len(yvalues) < 2:
        return float("NaN")
    
    # Create and standardize x values in one step
    x = np.arange(len(yvalues))
    x_std = (x - x.mean()) / x.std()
    
    # Calculate slope and round to 4 decimal places
    return round(np.polyfit(x_std, yvalues, 1)[0], 4)


def fn_average_trend(df, period, trend_period=None, nobaseline=False):
    """
    calculate the average rate of change within a windows of x years (the trend_period)
    """
    if trend_period is None:
        trend_period = period
    elif len(trend_period) > len(period):
        raise Exception(
            "trend_window_width must be less than or equal to year_window_width"
        )
    # calculate mean and slope of the 5years values difference from baseline standardised by the stddev

    if nobaseline:
        tmp = df[period]
    else:
        tmp = df[period].subtract(df["baseline"], axis=0).div(df["stddev"], axis=0) + 0
    # tmp=tmp.dropna()
    # sites with no variation (eg always dry have baseline=0, stddev=0 so standardised metric becomes na from divide by zero - recast to zero
    tmp = tmp.fillna(0)


    # debug print ('period', period)
    slope = "Trend" + str(period[-1])
    ave = "Ave" + str(period[-1])  # capital A ensures column sorts first
    tmp[ave] = tmp[period].mean(axis=1, numeric_only=True)
    cols = [ave]
    # Calculate slope if more than one period
    if len(period) > 1:
        # print ('trend_period', trend_period)
        # print(tmp)
        tmp[slope] = tmp[trend_period].apply(fn_slope, axis=1)
        cols.append(slope)
    # debug tmp.to_csv(str(year)+"fn_average_trend_tmp.csv")
    return tmp[cols]


# def aggregate_fields (ANAE, aggshp, aggfield):
#     agg = gpd.read_file(aggshp).to_crs("EPSG:3577")[aggfield + ['geometry']]
#     grp_labels = gpd.sjoin(ANAE, agg, how="left", op='intersects')[['UID', 'ANAE_TYPE','Area_Ha'] + aggfield].set_index('UID')
#     return grp_labels, aggfield


# def aggregate_area_weighted(df, agg, aggfield, ANAE = ANAE, ANAEgrp = [], pkey='UID'):
def aggregate_area_weighted(df, agg, aggfield, ANAEgrp=[]):
    """
    Takes parameter values for individual ANAE ecosystem polygons and aggregates the values to larger areas
    using an area weighting.  e.g. to aggregate a metric across all the ANAE polygons within a Ramsar site
    inputs: dataframe of parameter values per ANAE polygon
            a specified aggregator (one or more larger subunits that contain multiple ANAE polygons)

    Output is a single metric value for each larger area subunit calculated the area weighted mean of the ANAE polygons within it
    """
    cols = df.columns.values.tolist()
    # debug df.to_csv("fn_aggregate_area_weighted_df.csv")
    # debug grp_labels.to_csv("fn_aggregate_area_weighted_grp_labels.csv")

    agdf = df.join(agg, how="inner").replace(
        [np.inf, -np.inf], np.nan
    )  # there are some stray "inf" values from dividing by very small small stddev -covert to NaN so sum(numeric_only = True) can ignore them

    # debug agdf.to_csv("fn_aggregate_area_weighted_agdf.csv")
    if agg.name == "ANAE":
        return agdf[["grp"] + cols + ["Area_Ha"]].set_index("grp", append=True)
    else:
        agdf[cols] = agdf[cols].multiply(agdf["Area_Ha"], axis=0)
        # debug agdf.to_csv("fn_aggregate_area_weighted_agdftimesArea.csv")
        agg_data = (
            agdf[cols + ["Area_Ha"] + aggfield + ANAEgrp]
            .groupby(aggfield + ANAEgrp)
            .sum(numeric_only=True)
        )
        # debug agg_data.to_csv("fn_aggregate_area_weighted_agg_data.csv")
        agg_data[cols] = agg_data[cols].div(agg_data["Area_Ha"], axis=0)
        return agg_data[cols + ["Area_Ha"]]


def bin_stress_scores(grp_df, thresholds):
    # (thresholds, colname) = params
    """
    bin values into 3,2,1 (low, medium high) = reverse of condition binning
    applying pre-determined thresholds mapped in tsl_score dict.
    """
    _bins = thresholds[grp_df.name] + [float("inf")]
    # debug print (x['grp'].iat[0],_bins,list(range(len(_bins)-1,0,-1)))
    return pd.cut(
        grp_df, bins=_bins, right=False, labels=range(len(_bins) - 1, 0, -1)
    ).astype(
        "float"
    )  # return as float instead of category so we can multiply by area to scale up


def rename_stats_columns(c, metric_name):
    """
    a clumsy routine in ever evolving code to rename column headers in the data frame
    """
    c = c.replace("Ave", metric_name)
    c = c.replace("Trend", "T" + metric_name)
    return c


def deviation_from_baseline(
    df, metric, year_window_width, trend_window_width, nobaseline=False
):
    """
    calculate the deviation from the baseline in each year of the data frame

    append also the Trend in the deviations over the trend_window_width with prefix "T" on columns headings

    append also the SUM deviation over the year_window_width with prefix "sum" on columns headings (e.g. sum of the preceding 5 years)
    Args:
        df (pd.DataFrame): Input data with 'year' column
        metric (str): Name of metric column
        year_window_width (int): Size of moving window in years
        trend_window_width (int): Number of years for trend calculation
        nobaseline (bool): Skip baseline normalization if True
        
    Returns:
        pd.DataFrame: Metrics including deviations and trends
    """
    
    if year_window_width < 1:
        raise ValueError("year_window_width must be positive")
    if trend_window_width > year_window_width:
        raise ValueError("trend_window_width cannot exceed year_window_width")
    if df.empty:
        raise ValueError("Input DataFrame is empty")
    if "year" not in df.columns:
        raise KeyError("DataFrame must contain 'year' column")
    
    allyears = df["year"].unique().tolist()

    _years = range(allyears[0] + year_window_width - 1, allyears[-1] + 1)
    metric_name = metric.replace("+", "").lower()

    pivot = pivot_year(df, metric, pkey).round(4)
    pivot.to_csv(os.path.join(out_path, f"BWS_pivot_{metric}.csv"))
    dfs = []
    for p, year in enumerate(
        tqdm(
            _years,
            desc=f"{metric} in {year_window_width}y window, trend over {trend_window_width}y:",
        )
    ):
        period = list(range(year - year_window_width + 1, year + 1))
        trend_period = period[-trend_window_width:]
        stats_df = fn_average_trend(pivot, period, trend_period, nobaseline=nobaseline)
        # debug stats_df.to_csv(str(year)+"testmetrics.csv")
        # debug print(year,stats_df)
        # 
        # if we want to score before aggregating
        # ave = stats_df.columns.values.tolist()[0]
        # stats_df['sc'+metric+str(year)] = pd.cut(stats_df[ave], bins = _bins, labels = range(1,len(_bins))).astype('float') #float not default category so can be aggregated
        if len(stats_df.columns) > 1:
            stats_df["sum" + metric_name + str(year)] = stats_df.sum(axis=1)
            
            # if we want to score before aggregating
            # trend = stats_df.columns.values.tolist()[1]
            # stats_df['scT'+metric+str(year)] = pd.cut(stats_df[ave], bins = _bins, labels = range(1,len(_bins))).astype('float') #float not default category so can be aggregated
        dfs.append(stats_df)
    # aggregate all the metrics into a single data frame
    metrics_df = pd.concat(dfs, axis=1).sort_index(axis=1)
    metrics_df = metrics_df.rename(
        columns={
            c: rename_stats_columns(c.strip(), metric_name) for c in metrics_df.columns
        }
    )
    return metrics_df


def append_metric_scores(df, col_list, _bins, _labels):
    """
    scores the input data frame into bins and appends the scores for each year as
    additional columns added to the right edge of the data frame so the data can be easily viewed in Excel

    """
    dfs = []
    dfs.append(df)
    for col in col_list:
        dfs.append(
            pd.cut(df[col], bins=_bins, labels=_labels, include_lowest=True)
            .astype("float")
            .rename("sc" + col)
        )  # defaults to 'category' so recast to float so scores can be aggregated
    return pd.concat(dfs, axis=1)


def calc_pivots(
    df,
    metrics,
    year_window_width=5,
    trend_window_width=None,
    _bins=[float("-inf"), -1, 0, float("inf")],
    reverse_scores=False,
    nobaseline=False,
    tag="",
):
    """
    This brings together some of the code above to
    summarise the metrics in a moving window of multiple years
    (for the BWS vulnerabilities project the veg condition and trend (rate of change) over a
    moving 5 years period was calculated for each year of the data frame

    The metrics are then aggregated to the pre-defined larger spatial subunits (Ramsar sites, waterbird breeding sites, valleys)

    Pivot tables are written to the working directory as cvs files for inspection in Excel.
    The pivot tables will also be read in and scored for the final integration of vulnerability metrics

    reverse_scores true/false is used to switch the logic for different metrics
    e.g. more green veg (WIT pv) is good, more bare soil (WIT bs) is bad

    """
    if isinstance(metrics, str):
        metrics = [metrics] # ensure metrics is a list
    if not isinstance(_bins, list):
        raise TypeError("bins must be a list of thresholds")

    if trend_window_width is None:
        trend_window_width = year_window_width
    elif trend_window_width > year_window_width:
        raise Exception(
            "trend_window_width must be less than or equal to year_window_width"
        )
    _scores = list(range(1, len(_bins)))
    if reverse_scores:
        _scores = _scores[::-1]
    wit = {}
    for metric in metrics:
        metrics_df = deviation_from_baseline(
            df, metric, year_window_width, trend_window_width, nobaseline=nobaseline
        )
        print("Aggregate metrics and scores:")
        for ag in aggregators:
            fname = f"{metric}_{ag}_{year_window_width}yr{tag}.csv"
            print(f"     {ag} - {fname}")
            wit[ag] = aggregate_area_weighted(
                metrics_df, aggregators[ag], aggfield[ag], ANAEgrp=["grp"]
            ).round(4)
            col_list = [c for c in wit[ag].columns.values.tolist() if c != "Area_Ha"]
            wit[ag] = append_metric_scores(wit[ag], col_list, _bins, _scores)
            wit[ag].to_csv(os.path.join(out_path, fname))


def normalise(df):
    """
    normalises an input data frame to values between 0-1
    used to normalise NDVI from older NOAA AVHRR and newer MODIS
    """
    dmin = df.min()
    drange = df.max() - dmin
    return df.subtract(dmin).divide(drange)


def standardise_z(df):
    """
    z-score standardise an input data frame
    used to standardise NDVI from older NOAA AVHRR and newer MODIS
    """
    dmean = df.mean()
    dstdev = df.std()
    return df.subtract(dmean).divide(dstdev)


def extract_scores(index_cols, fname, score_cols):
    """
    Retrieves the score columns from specified output pivot tables
    and strips off the string prefix from year columns.
    This standardises the format of the different score tables so metrics
    can be summed and counted across groups and features
    with pandas append and groupby functions
    """
    # print(f'     Reading scores from {fname}...') #DEBUG
    scores = pd.read_csv(fname, low_memory=False)
    scores = scores[index_cols + score_cols].set_index(index_cols)
    return scores.rename(columns={c: c[-4:] for c in scores.columns})


def normalise(df):
    """
    Normalizes values in a DataFrame or Series to a scale of 0 to 1 using min-max normalization.

    Parameters
    ----------
    df : pandas.DataFrame or pandas.Series
        If the input is a DataFrame, the global minimum and maximum across all columns are used for normalization.

    Returns
    -------
    pandas.DataFrame or pandas.Series
        Normalized data with values scaled between 0 and 1.
    """
    df_min = df.min()
    if not pd.api.types.is_scalar(df_min):
        df_min = df_min.min()
    df_max = df.max()
    if not pd.api.types.is_scalar(df_max):
        df_max = df_max.max()
    return df.subtract(df_min).divide(df_max - df_min)


def sum_and_normalise_weighted(df):
    """
    sums the condition/stress scores for each feature in the index and rescales the data
    normalising to range 0-1 allowing for cells with missing data
    rescaled = (sum - count) / (count * number of possible metrics) - count)
    """
    sum_df = df.groupby(level=df.index.names).sum()
    count_df = df.groupby(level=df.index.names).count()
    return normalise(sum_df.divide(count_df))

In [5]:
# initialise data stores  (names are data frames used in the code)
# this allows cells in the notebook to be re-run quickly without re-reading all the iput data multiple times when we dont have to

ANAE = None
Valley = None
DIWA = None
Ramsar = None

soil_moisture_df = None
wit_yearly_metrics_df = None
wit_time_since_last_inundation_df = None
wit_inundation_metrics_df = None
ndvi_df = None

# Load in the spatial subunits - ANAE and aggregating layers

This block of code is slow to run as it:
1. first reads in the ANAE polygons
2. spatially join the ANAE to multiple data sets with larger-scale subunits to map the aggregations of individual ANAE polgons required to represent larger areas (e.g. Ramsar sites, DIWA wetlands, Valleys

The spatial data is read in once and stored and will not be re-read if the cell is re-run.  If the data needs to be read again either reset the notebook at start again or re-run the cell above that initialises the data stores to NONE.

In [6]:
# name of the unique ID identifying each ANAEv3 polygon is a 9 character geohash
pkey = "UID"

# dictionary of the aggregators - for the BWS vulnerabilites project we were interested in scaling up from ANAE polygons to larger subunits including


# DTwaterbirds =  dirty thirty =  MDBA waterbird units
# DIWA = Directory of Important Wetlands
# Ramsar
# Valleys are valleys used by MDBA uses as "vegetation regions" to guide priority setting in the BWS
# Basin is the whole MDB


# aggfield is the unique identifier for subunits within each of the aggregator data sets
# note for Ramsar sites we can aggregate ANAE polygons to multiple "wetlands" within each ramsar site
aggregators = {
    "ANAE": ANAE,
}

aggfield = {
    "ANAE": ["UID"],
    "Basin": [],
}

# spatial data is read into geopandas data frames

# if we havn't read the ANAE data in before the do it now reading into a geopandas frame
# for the ANAE we simplify the typology by grouping some of the ANAE ecosystem types with the same dominant vegetation
# e.g. we take black box floodplains and blackbox woodland swamps and combine into a single "black box" class

ANAE_shp = os.path.join(spatial_path, "ANAEv3_BWS.shp")
Valley_shp = os.path.join(spatial_path, "BWSRegions.shp")
DIWA_shp = os.path.join(spatial_path, "DIWA_complex.shp")
Ramsar_shp = os.path.join(spatial_path, "ramsar_wetlands.shp")

# -----------------------------------------------------------------
# ANAE scale
# -----------------------------------------------------------------
def assign_group_name(anae_type):
    anae_type = anae_type.lower()
    if "river red gum" in anae_type and "woodland" in anae_type:
        return "river red gum woodland"
    elif "river red gum" in anae_type:
        return "river red gum swamps and forests"
    elif "black box" in anae_type:
        return "black box"
    elif "coolibah" in anae_type:
        return "coolibah"
    elif "lignum" in anae_type:
        return "lignum"
    elif "cooba" in anae_type:
        return "cooba"
    elif "f2.4: shrubland riparian zone or floodplain" in anae_type:
        return "shrubland"
    #removed permanent lakes as not vegetated
    # elif (
    #     "permanent lake" in anae_type
    #     or "permanent wetland" in anae_type
    #     or "aquatic bed" in anae_type
    # ):
    #     return "submerged lake"
    elif "tall emergent marsh" in anae_type:
        return "tall reed beds"
    elif "grass" in anae_type or "meadow" in anae_type:
        return "grassy meadows"
    elif (
        "forb marsh" in anae_type
        or "temporary wetland" in anae_type
        or "temporary lake" in anae_type
    ):
        return "herbfield"
    #removed clay pans as not vegetated
    # elif "clay" in anae_type:
    #     return "clay pan"
    # else:
    #     return None  # optional fallback





if not os.path.exists(ANAE_shp):
    raise FileNotFoundError(f"ANAE shapefile not found: {ANAE_shp}")
if ANAE is None:
    print("Reading ANAE...")
    ANAE = gpd.read_file(os.path.join(spatial_path, "ANAEv3_BWS.shp")).to_crs(
        "EPSG:3577"
    )
    #discard all the attribute columns apart from those listed
    ANAE = ANAE[["UID", "ANAE_TYPE", "Area_Ha", "geometry"]]
    
    #define the vegetation functional groups using the ANAE types
    ANAE["grp"] = ANAE["ANAE_TYPE"].apply(assign_group_name)
    ANAE.name = "ANAE"  # used to area-weight aggregate all ANAE polygons

else:
    print("ANAE already read in")


aggregators = {
    "ANAE": ANAE,
}

# -----------------------------------------------------------------
# Basin scale
# -----------------------------------------------------------------
aggregators["Basin"] = ANAE[["grp", "Area_Ha"]]
aggregators["Basin"].name = "Basin"

# -----------------------------------------------------------------
# Valley scale
# -----------------------------------------------------------------

if Valley is None and os.path.exists(Valley_shp):
    print("Reading BWS valleys...")
    aggfield["Valley"] = ["BWS_Region"]
    Valley = gpd.read_file(Valley_shp).to_crs("EPSG:3577")
    aggregators["Valley"] = (
        gpd.sjoin(
            ANAE,
            Valley[aggfield["Valley"] + ["geometry"]],
            how="left",
            predicate="intersects",
        )
        .dropna()
        .set_index("UID")
    )
    aggregators["Valley"].name = "Valley"


# Load Aggregator shapefiles and use spatial joins to determine the ANAE polygons
# that are within each larger spatial unit. In theory we could have saved these
# as lookup tables for speedy re-use but left coded this way for possible flexibility in the future
# if waterbird boundaries or Ramsar boundaries change

# -----------------------------------------------------------------
# Directory of Important Wetlands (DIWA)
# -----------------------------------------------------------------
if DIWA is None and os.path.exists(DIWA_shp):
    print("Reading DIWA...")
    aggfield["DIWA"] = ["WNAME"]
    DIWA = gpd.read_file(DIWA_shp).to_crs("EPSG:3577")
    aggregators["DIWA"] = (
        gpd.sjoin(
            ANAE,
            DIWA[aggfield["DIWA"] + ["geometry"]],
            how="left",
            predicate="intersects",
        )
        .dropna()
        .set_index("UID")
    )
    aggregators["DIWA"].name = "DIWA"
    

# -----------------------------------------------------------------
# Ramsar wetlands
# -----------------------------------------------------------------

if Ramsar is None and os.path.exists(Ramsar_shp):
    print("Reading Ramsar...")
    aggfield["Ramsar"] = ["RAMSAR_NAM", "WETLAND_NA"]
    Ramsar = gpd.read_file(Ramsar_shp).to_crs("EPSG:3577")
    aggregators["Ramsar"] = gpd.sjoin(
            ANAE,
            Ramsar[aggfield["Ramsar"] + ["geometry"]],
            how="left",
            predicate="intersects",
        ).dropna().set_index("UID")
    
    aggregators["Ramsar"].name = "Ramsar"
    


if ANAE.index.name != "UID":
    ANAE = ANAE.set_index("UID")
    ANAE.name = "ANAE"
aggregators["ANAE"] = ANAE

# aggregators = {
#     "ANAE": ANAE,
#     "DIWA": DIWA,
#     "Ramsar": Ramsar,
#     "Valley": Valley,
#     "Basin": Basin,
# }

Reading ANAE...
Reading BWS valleys...
Reading DIWA...
Reading Ramsar...


In [7]:
# this is a debug check to list the ANAE types that are NOT assigned to a functional group to see if we missed anything obvious
# the types that list here are the types we *don't* include in the determination of wetland/floodplain/waterbird vulnerabilities
# (e.g. includes rivers and streams, saline systems)

ANAE[ANAE["grp"].isna()]["ANAE_TYPE"].unique()



array(['F4: Unspecified riparian zone or floodplain',
       'Etd1.2.1: Tide dominated saltmarsh', 'Pt3.1.2: Clay pan',
       'Lp1.1: Permanent lake', 'Etd1.3.3: Tide dominated estuary',
       'Ewd1.2.3: Intertidal saltmarsh',
       'Ewd1.2.4: Intertidal mudflat or sand bar',
       'Pt1.8.2: Temporary shrub swamp', 'Ewd1.3.2: Coastal lagoon',
       'Etd1.2.2: Tide dominated mudflats and sandbar',
       'Pp4.2: Permanent wetland', 'Pst4: Temporary saline wetland',
       'Pst2.2: Temporary salt marsh', 'Etd1.2.3: Tide dominated forest',
       'Rp1.4: Permanent lowland stream',
       'Rt1.4: Temporary lowland stream',
       'Pst1.1: Temporary saline swamp',
       'Etd1.1.1: Tide dominated rocky shoreline',
       'Ewd1.2.5: Intertidal rocky shoreline',
       'Pt1.5.2: Temporary paperbark swamp',
       'Rt1.2: Temporary transitional zone stream',
       'Pt1.6.2: Temporary woodland swamp',
       'F1.13: Paperbark riparian zone or floodplain',
       'Lsp1.1: Permanent saline 

# Process the WIT annual metrics
Read in the WIT yearly metrics and calculate pivot tables (note this is slow process but a progress bar is shown).  After the initial calculation of metrics per ANAE polygon is complete the data are aggregated to the various larger spatial scales.


**input:** WIT Yearly statistics generated by the wit_metrics notebook as file **RESULT_ANAE_yearly_metrics.csv**

**output:** pivot tables for each WIT metric x spatial aggregator combination as CSV files in the working directory


In [8]:
#-----------------------------------------------------------------
# Note change - data is now read directly from the zip file to avoid 
# reading unintentionally modified files and to save storage space
#-----------------------------------------------------------------

if wit_yearly_metrics_df is None:  # read the data in if hasn't been read in already
    # use geopandas instead of pandas so we can later call on the 'name' attribute which is not in the version of pandas were using
    print("Reading WIT metrics...")
    wit_yearly_metrics_df = pd.read_csv(
        os.path.join(data_path, "RESULT_WIT_ANAE_yearly_metrics.zip")
    ).rename(columns={"feature_id": "UID"})
    # remove all records not on the managed floodplain
    wit_yearly_metrics_df = wit_yearly_metrics_df[wit_yearly_metrics_df["UID"].isin(ANAE.index)]

# debug code = limit to first 1000 records so it runs quickly
# wit_yearly = wit_yearly.head(1000)

allyears = (
    wit_yearly_metrics_df["year"].unique().tolist()
)  # list of all years included in the data frame
baseline = [
    y for y in allyears if y not in millennium_drought
]  # alltime excluding the millennium_drought


# we pass a subset of metrics that we are interested in to the "calc_pivots" function
# for the BWS project we calculated vegetation stress metrics in a 5year moving window looking at the trend
# (rate and direction of change) within the most recent  the last 2 years
# the summary pivot tables for each metric x aggregator combination are written to the working directory as CSV files

metrics = ["water+wet_median", "pv_median", "npv_median", "npv+pv+wet_median"]

# water+wet_median represents inundation
# pv_median =  fractional cover of green vegetation
# npv =  fractional cover of non-green (brown) vegetation
# npv+pv+wet_median = all vegetation (green, brown and wet vegetation)


# npv+pv+wet =  combines all WIT vegetation, water+wet_median represents inundation
calc_pivots(
    wit_yearly_metrics_df,
    metrics,
    year_window_width=veg_window_width,
    trend_window_width=veg_trend_width,
)

# median bare soil (bs_median) is processed separately with the reverse_scores=True switch to reverse
# because more bare soil represents declining condition

# bare soil 
calc_pivots(
    wit_yearly_metrics_df,
    "bs_median",
    year_window_width=veg_window_width,
    trend_window_width=veg_trend_width,
    reverse_scores=True,
)

# not used - keep for reference
# waterbirds respond quicker than trees so the BWS project used annual metrics (year_window_width=1) and there is no multi-year trend
# metrics = ["water+wet_median", "pv_median"]
# calc_pivots(wit_yearly, metrics, year_window_width=1)

Reading WIT metrics...


water+wet_median in 5y window, trend over 2y::   0%|          | 0/36 [00:00<?, ?it/s]

Aggregate metrics and scores:
     ANAE - water+wet_median_ANAE_5yr.csv
     Basin - water+wet_median_Basin_5yr.csv
     Valley - water+wet_median_Valley_5yr.csv
     DIWA - water+wet_median_DIWA_5yr.csv
     Ramsar - water+wet_median_Ramsar_5yr.csv


pv_median in 5y window, trend over 2y::   0%|          | 0/36 [00:00<?, ?it/s]

Aggregate metrics and scores:
     ANAE - pv_median_ANAE_5yr.csv
     Basin - pv_median_Basin_5yr.csv
     Valley - pv_median_Valley_5yr.csv
     DIWA - pv_median_DIWA_5yr.csv
     Ramsar - pv_median_Ramsar_5yr.csv


...

Aggregate metrics and scores:
     ANAE - npv_median_ANAE_5yr.csv
     Basin - npv_median_Basin_5yr.csv
     Valley - npv_median_Valley_5yr.csv
     DIWA - npv_median_DIWA_5yr.csv
     Ramsar - npv_median_Ramsar_5yr.csv


npv+pv+wet_median in 5y window, trend over 2y::   0%|          | 0/36 [00:00<?, ?it/s]

Aggregate metrics and scores:
     ANAE - npv+pv+wet_median_ANAE_5yr.csv
     Basin - npv+pv+wet_median_Basin_5yr.csv
     Valley - npv+pv+wet_median_Valley_5yr.csv
     DIWA - npv+pv+wet_median_DIWA_5yr.csv
     Ramsar - npv+pv+wet_median_Ramsar_5yr.csv


bs_median in 5y window, trend over 2y::   0%|          | 0/36 [00:00<?, ?it/s]

Aggregate metrics and scores:
     ANAE - bs_median_ANAE_5yr.csv
     Basin - bs_median_Basin_5yr.csv
     Valley - bs_median_Valley_5yr.csv
     DIWA - bs_median_DIWA_5yr.csv
     Ramsar - bs_median_Ramsar_5yr.csv


# Process the WIT Time since last inundation
Read in the WIT time since last inundation metrics and score the stress using the user defined thresholds for HIGH, MEDIUM and LOW stress that are defined in the code above (vegetation_tsli_score)

**input:** WIT Time since last inundation statistics generated by the wit_metrics notebook as file **'RESULT_WIT_ANAE_time_since_last_inundation.csv**

**output:** pivot tables for each spatial aggregator with the average time since last inundation in each calendar year for different vegetation groupings scored on a scale of 1-3

In [9]:
#-----------------------------------------------------------------
# Note change - data is now read directly from the zip file to avoid 
# reading unintentionally modified files and to save storage space
#-----------------------------------------------------------------

def score_tsli(df, name, thresholds):
    """
    bins the time since last inundation metrics per polygon per year in df
    into scores (1,2,3) appending the scores to the data frame
    then aggregates the ANAE polygon score to the larger spatial scales
    
    """
    # print(f"score time since last inundation for {name}")
    cols=[]
    scores = []
    for y in tqdm(alltime, desc=f"Score time since last inundation for {name}"):
        col = f"tsli{y}"
        cols.append(col)
        score = f"sc_tsli{y}"
        scores.append(score)
        df[score] = df.groupby("grp")[col].apply(
        bin_stress_scores, thresholds)

    cols = cols + scores
    print("Aggregate metrics and scores:")
    for ag in aggregators:
        fname = f"time_since_last_inundation_{ag}_{name}.csv"
        print(f"     {ag} - {fname}")
        tsli_ag = aggregate_area_weighted(
            df[cols], aggregators[ag], aggfield[ag], ANAEgrp=["grp"]
        )
        tsli_ag.round(1).to_csv(os.path.join(out_path, fname))


if wit_time_since_last_inundation_df is None:  # read the data in if required otherwise re-use
    print("Reading WIT time since last inundation...")
    wit_time_since_last_inundation_df = (
        pd.read_csv(
            os.path.join(data_path, "RESULT_WIT_ANAE_time_since_last_inundation.zip"),
            parse_dates=["end_date", "final_date"],
        )
        .rename(columns={"feature_id": "UID"})
        .set_index("UID")
    )


if wit_inundation_metrics_df is None:
    print("Reading WIT inundation metrics...")
    wit_inundation_metrics_df = pd.read_csv(
        os.path.join(data_path, "RESULT_WIT_ANAE_inundation_metrics.zip"),
        parse_dates=["start_date", "end_date"],
    ).rename(
        columns={"feature_id": "UID"}
    ) 
    
    # much faster to convert these data types once the dataframe is loaded into pandas compared to using converters on csv read
    wit_inundation_metrics_df["duration"] = pd.to_timedelta(wit_inundation_metrics_df["duration"]).dt.days
    wit_inundation_metrics_df["gap"] = pd.to_timedelta(wit_inundation_metrics_df["gap"]).dt.days


tsli_df = wit_time_since_last_inundation_df.join(ANAE[["ANAE_TYPE", "grp", "Area_Ha"]])



for y in tqdm(alltime, desc="Time since last inundation per year"):
    cutoff_date = pd.to_datetime(str(y) + "-12-31")

    idf = wit_inundation_metrics_df[wit_inundation_metrics_df["start_date"] < cutoff_date].copy()
    idf.loc[idf["end_date"] > cutoff_date, "end_date"] = cutoff_date
    idf.loc[idf["end_date"] == cutoff_date, "duration"] = (
        idf["end_date"] - idf["start_date"]
    ).dt.days

    timesincelast = (
        idf[(idf["end_date"] == idf.groupby("UID")["end_date"].transform("max"))]
        .copy()
        .set_index("UID")
    )
    # timesincelast=timesincelast.join(tsli_df['final-date'].astype('datetime64[D]')) #causes and error in newer python  - use to_datetime instead
    timesincelast = timesincelast.join(tsli_df["final_date"])
    timesincelast.loc[timesincelast["final_date"] > cutoff_date, "final_date"] = (
        cutoff_date
    )
    # tsli for this year
    tsli_df[f"tsli{y}"] = (
        timesincelast["final_date"] - timesincelast["end_date"]
    ).dt.days

# time since last inundation of different vegetation functional groups scored by user defined thresholds
score_tsli(tsli_df, "vegetation_stress", vegetation_tsli_stress_thresholds)



Reading WIT time since last inundation...
Reading WIT inundation metrics...


Time since last inundation per year:   0%|          | 0/38 [00:00<?, ?it/s]

Score time since last inundation for vegetation_stress:   0%|          | 0/38 [00:00<?, ?it/s]

Aggregate metrics and scores:
     ANAE - time_since_last_inundation_ANAE_vegetation_stress.csv
     Basin - time_since_last_inundation_Basin_vegetation_stress.csv
     Valley - time_since_last_inundation_Valley_vegetation_stress.csv
     DIWA - time_since_last_inundation_DIWA_vegetation_stress.csv
     Ramsar - time_since_last_inundation_Ramsar_vegetation_stress.csv


## Process NDVI
Data obtained from two different Google Earth Engine data sets required to represent the full time period

* 1986_2000 - NOAA AVHRR  https://developers.google.com/earth-engine/datasets/catalog/NOAA_CDR_AVHRR_NDVI_V5

* 2001_2024 - MODIS https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MOD13Q1



The data sets are not directly comparable with NDVI values obtained from AVHRR being approx 50% of MODIS
(a function of the data ranges as provided in Google's earth engine data library).

Each data set is therefore normalised to range 0-1 before appending them
Experimental - have included standardising the two data sets (dividing by standard deviation) or z-scores

**input:** average NDVI per ANAE polygon per year as CSV files **NDVI_1986-2000_ANAEv3_annual_AVHRR.csv** and **NDVI_2001-2024_ANAEv3_annual_MODIS.csv**
**output:** NDVI pivot tables for each spatial aggregation  as CSV files in the working directory

In [10]:
#-----------------------------------------------------------------
# Note change - data is now read directly from the zip file to avoid 
# reading unintentionally modified files and to save storage space
#-----------------------------------------------------------------

if ndvi_df is None:  # read the data in only once if the cell is re-run
    print("Reading AVHRR_NDVI data...")
    # NOAA_CDR_AVHRR_NDVI_V5 range -9998 to 9998
    ndvi1 = pd.read_csv(
        os.path.join(data_path, "NDVI_1986-2000_ANAEv3_annual_AVHRR.zip"),
        dtype={"UID": str, "year": int, "NDVI": float},
    )
    ndvi1["NDVI"] = normalise(ndvi1["NDVI"])

    ndvi_z1 = ndvi1.copy()

    ndvi_z1["NDVI"] = standardise_z(ndvi1["NDVI"])  # as anomaly

    # MODIS_061_MOD13Q1 range -9998 to 10000
    print("Reading MODIS_NDVI data...")
    ndvi2 = pd.read_csv(
        os.path.join(data_path, "NDVI_2001-2024_ANAEv3_annual_MODIS.zip"),
        dtype={"UID": str, "year": int, "NDVI": float},
    )
    ndvi2["NDVI"] = normalise(ndvi2["NDVI"])
    ndvi_z2 = ndvi2.copy()
    ndvi_z2["NDVI"] = standardise_z(ndvi1["NDVI"])

    ndvi_df = pd.concat([ndvi1, ndvi2], ignore_index=True)
    ndvi_std_df = pd.concat([ndvi_z1, ndvi_z2], ignore_index=True)

allyears = (
    ndvi_df["year"].unique().tolist()
)  # list of all years included in the data frame
baseline = [
    y for y in allyears if y not in millennium_drought
]  # allyears excluding the millennium_drought

# we pass a subset of metrics that we are interested in to the "calc_pivots" function
# for the BWS project we calculated vegetation stress metrics in a 5year moving window looking at the trend
# (rate and direction of change) within the most recent  the last 2 years
# the summary pivot tables for each metric x aggregator combination are written to the working directory as CSV files

#normalised NDVI is used as a vegetation condition metric as anomaly from the baseline mean
#values rescaled to 0-1 to harmonise NDVI from the older NOAA AVHRR with the newer MODIS NDVI
calc_pivots(
    ndvi_df,
    "NDVI",
    year_window_width=veg_window_width,
    trend_window_width=veg_trend_width,
)

#standardised NDVI is used as a vegetation condition metric as anomaly from the baseline mean
# values standardised by dividing by the standard deviation to harmonise NDVI from the older NOAA AVHRR with the newer MODIS NDVI
calc_pivots(
    ndvi_std_df,
    "NDVI",
    year_window_width=veg_window_width,
    trend_window_width=veg_trend_width,
    nobaseline=True,
    tag="_standardised",
)

# annual NDVI (not used for vegetation here but included for reference in output data)
# this is the annual NDVI without the moving window or trend
calc_pivots(
    ndvi_df,
    "NDVI",
    year_window_width=1,  # annual NDVI
)

Reading AVHRR_NDVI data...
Reading MODIS_NDVI data...


NDVI in 5y window, trend over 2y::   0%|          | 0/35 [00:00<?, ?it/s]

Aggregate metrics and scores:
     ANAE - NDVI_ANAE_5yr.csv
     Basin - NDVI_Basin_5yr.csv
     Valley - NDVI_Valley_5yr.csv
     DIWA - NDVI_DIWA_5yr.csv
     Ramsar - NDVI_Ramsar_5yr.csv


NDVI in 5y window, trend over 2y::   0%|          | 0/35 [00:00<?, ?it/s]

Aggregate metrics and scores:
     ANAE - NDVI_ANAE_5yr_standardised.csv
     Basin - NDVI_Basin_5yr_standardised.csv
     Valley - NDVI_Valley_5yr_standardised.csv
     DIWA - NDVI_DIWA_5yr_standardised.csv
     Ramsar - NDVI_Ramsar_5yr_standardised.csv


NDVI in 1y window, trend over 1y::   0%|          | 0/39 [00:00<?, ?it/s]

Aggregate metrics and scores:
     ANAE - NDVI_ANAE_1yr.csv
     Basin - NDVI_Basin_1yr.csv
     Valley - NDVI_Valley_1yr.csv
     DIWA - NDVI_DIWA_1yr.csv
     Ramsar - NDVI_Ramsar_1yr.csv


# Process Soil Moisture

In [11]:
#-----------------------------------------------------------------
# Note change - data is now read directly from the zip file to avoid 
# reading unintentionally modified files and to save storage space
#-----------------------------------------------------------------
# root zone soil moisture is used as a vegetation stress metric

if soil_moisture_df is None:  # read the data in if required otherwise re-use
    print("Reading root zone soil moisture data...")
    soil_moisture_df = pd.read_csv(
        os.path.join(data_path, "ZonalSt30_soilmoistureanomally.zip")
    ).rename(columns={"MEAN": "soilmoist"})
    soil_moisture_df["year"] = pd.to_datetime(soil_moisture_df["StdTime"]).dt.year


# root zone soil moisture is derived from the soil moisture relative data set https://awo.bom.gov.au/products/historical/soilMoisture-rootZone
# - with values 0 to 1 which is percentile rank over the average (2011-2017) baseline
# use different bins to set scoring _bins=[-1, 0.25, 0.5, 1] which results in:
# soil moisture <0.25 = score 1   (less than 25% of values above baseline  or 75% below baseline)
# soil moisture value 0.25-0.5 = score 2  (only 25%-50% above baseline)
# soil moisture value > 0.5 = score 3  (50% of values above baseline)
# since the metric is already expressed as an anomaly we pass nobaseline = True to prevent calculation of baseline average

# we pass a subset of metrics that we are interested in to the "calc_pivots" function
# for the BWS project we calculated vegetation stress metrics in a 5year moving window looking at the trend
# (rate and direction of change) within the most recent  the last 2 years
# the summary pivot tables for each metric x aggregator combination are written to the working directory as CSV files

calc_pivots(
    soil_moisture_df,
    "soilmoist",
    year_window_width=veg_window_width,
    trend_window_width=veg_trend_width,
    _bins=[-1, 0.25, 0.5, 1],
    nobaseline=True, 
)


# annual year soil moisture (not used for vegetation here but included for reference in output data)
calc_pivots(
    soil_moisture_df,
    "soilmoist",
    year_window_width=1,
    _bins=[-1, 0.25, 0.5, 1],
    nobaseline=True,
)

Reading root zone soil moisture data...


soilmoist in 5y window, trend over 2y::   0%|          | 0/35 [00:00<?, ?it/s]

Aggregate metrics and scores:
     ANAE - soilmoist_ANAE_5yr.csv
     Basin - soilmoist_Basin_5yr.csv
     Valley - soilmoist_Valley_5yr.csv
     DIWA - soilmoist_DIWA_5yr.csv
     Ramsar - soilmoist_Ramsar_5yr.csv


soilmoist in 1y window, trend over 1y::   0%|          | 0/39 [00:00<?, ?it/s]

Aggregate metrics and scores:
     ANAE - soilmoist_ANAE_1yr.csv
     Basin - soilmoist_Basin_1yr.csv
     Valley - soilmoist_Valley_1yr.csv
     DIWA - soilmoist_DIWA_1yr.csv
     Ramsar - soilmoist_Ramsar_1yr.csv


# Combine all Vegetation Scores



In [12]:
years = range(alltime[0] + veg_window_width - 1, alltime[-1] + 1)

cond = ["npv+pv+wet_median", "ndvi"]
stress = ["water+wet_median", "time_since_last_inundation", "SoilMoist"]


print(
    f"""
Condition is the sum of {len(cond)} metrics, {cond}
Stress is the sum of {len(stress)} metrics, {stress}

The scores from saved pivot files in out_path {out_path}"

Summed scores for condition and stress are rescaled to range 0-1 to allow for missing data in spatial features

Vulnerability = condition + stress (rescaled 0-1)

A matrix with condition | stress | vulnerability scores is output for each aggregator scale...
"""
)


# we process all aggregators so vulnerability can be assessed at any of the scales
# for veg the BWS regions are the 'valleys'


for ag in aggregators:
    fname = f"FINAL_BWSVulnerability_vegetation_{ag}.csv"
    index_cols = aggfield[ag] + ["grp"]
    print(f"     {ag} - {fname}")

    # -----------------------------------------------------------------
    #  CONDITION
    # -----------------------------------------------------------------

    # load in the scores from the condition metric pivot tables that are contributing to the vulnerability score
    # cond_a = extract_scores(ag, f"meanTSC_{ag}_5yr.csv", 'scsummeantsc')
    cond_a = extract_scores(
        index_cols,
        os.path.join(out_path, f"npv+pv+wet_median_{ag}_5yr.csv"),
        score_cols=[f"scsumnpvpvwet_median{y}" for y in years],
    )
    cond_b = extract_scores(
        index_cols,
        os.path.join(out_path, f"ndvi_{ag}_5yr.csv"),
        score_cols=[f"scsumndvi{y}" for y in years],
    )

    # stacking the three metrics vertically in a df with the same index allows us to use groupby sum and count by index feature

    combined_cond_df = pd.concat([cond_a, cond_b], axis=0)

    # the condition score is the sum of multiple individual metric scores (nominally three for vegetation in this case)
    # however can be sum of 2-3 metrics if there are missing data resulting in a lower sum.
    # The simple normalise function would penalise cells with fewer metrics contributing (thus lower score)
    # Therefore use a revised method that weights by the count of metrics contributing to each sum is used
    cond_df = sum_and_normalise_weighted(combined_cond_df).round(1)

    # -----------------------------------------------------------------
    #  STRESS
    # -----------------------------------------------------------------
    # load in the scores from the stress metric pivot tables that are contributing to the vulnerability score
    stress_a = extract_scores(
        index_cols,
        os.path.join(out_path, f"water+wet_median_{ag}_5yr.csv"),
        score_cols=[f"scsumwaterwet_median{y}" for y in years],
    )
    stress_b = extract_scores(
        index_cols,
        os.path.join(
            out_path, f"time_since_last_inundation_{ag}_vegetation_stress.csv"
        ),
        score_cols=[f"sc_tsli{y}" for y in years],
    )
    stress_c = extract_scores(
        index_cols,
        os.path.join(out_path, f"SoilMoist_{ag}_5yr.csv"),
        score_cols=[f"scsumsoilmoist{y}" for y in years],
    )

    # stacking the three metrics vertically in a df with the same index allows us to use groupby to sum and count by index feature
    combined_stress_df = pd.concat([stress_a, stress_b, stress_c], axis=0)

    # the stress score is the sum of multiple individual metric scores (nominally three for vegetation in this case)
    # however can be sum of 1-2 metrics if there are missing data resulting in a lower sum.
    # The simple normalise function would penalise cells with fewer metrics contributing (thus lower score)
    # Therefore use a revised method that weights by the count of metrics contributing to each sum is used
    stress_df = sum_and_normalise_weighted(combined_stress_df).round(1)

    # -----------------------------------------------------------------
    #  VULNERABILITY
    # -----------------------------------------------------------------

    # vulnerability is condition + stress normalised to range 0-1
    tmp_df = pd.concat([cond_df, stress_df], axis=0)
    vulnerability_df = normalise(tmp_df.groupby(tmp_df.index.names).sum()).round(1)

    # for compatibility with Excel for user data review having headers that are numbers (i.e. years) cause "issues"
    # rename the columns from numerical years to strings with cond, stress, vul prefix so that Excel treats them as headers

    cond_df = cond_df.rename(columns={c: f"cond{c}" for c in cond_df.columns})
    stress_df = stress_df.rename(columns={c: f"stress{c}" for c in stress_df.columns})
    vulnerability_df = vulnerability_df.rename(
        columns={c: f"vul{c}" for c in vulnerability_df.columns}
    )

    # append columns for condition, stress and vulnerability scores into a single table

    score_df = (
        pd.concat([cond_df, stress_df, vulnerability_df], axis=1)
        .sort_index(axis=1)
        .to_csv(os.path.join(out_path, fname))
    )

    # NOTE output scale used for vegetation in the report is Valley scale
    # The relevant output file is FINAL_BWSVulnerability_vegetation_Valley.csv


print("done.")


Condition is the sum of 2 metrics, ['npv+pv+wet_median', 'ndvi']
Stress is the sum of 3 metrics, ['water+wet_median', 'time_since_last_inundation', 'SoilMoist']

The scores from saved pivot files in out_path ./output/"

Summed scores for condition and stress are rescaled to range 0-1 to allow for missing data in spatial features

Vulnerability = condition + stress (rescaled 0-1)

A matrix with condition | stress | vulnerability scores is output for each aggregator scale...

     ANAE - FINAL_BWSVulnerability_vegetation_ANAE.csv
     Basin - FINAL_BWSVulnerability_vegetation_Basin.csv
     Valley - FINAL_BWSVulnerability_vegetation_Valley.csv
     DIWA - FINAL_BWSVulnerability_vegetation_DIWA.csv
     Ramsar - FINAL_BWSVulnerability_vegetation_Ramsar.csv
done.


#END